# MNIST Handwritten Digit Classification using Neural Networks

This notebook builds a moderate neural network project for recognizing handwritten digits from the MNIST dataset.

## 1. Import Dependencies

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import confusion_matrix

tf.random.set_seed(3)
np.random.seed(3)

## 2. Load the MNIST Dataset

In [ ]:
(X_train, Y_train), (X_test, Y_test) = keras.datasets.mnist.load_data()
print("Training data:", X_train.shape, Y_train.shape)
print("Testing data:", X_test.shape, Y_test.shape)

The MNIST dataset contains 60,000 training images and 10,000 testing images. Each image is grayscale and has a size of 28 x 28 pixels.

## 3. Display a Sample Image

In [ ]:
plt.imshow(X_train[25], cmap="gray")
plt.title(f"Label: {Y_train[25]}")
plt.axis("off")
plt.show()

## 4. Normalize the Image Data

In [ ]:
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0
print("Minimum pixel value:", X_train.min())
print("Maximum pixel value:", X_train.max())

## 5. Build the Neural Network

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(28, 28)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax")
])

model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.summary()

## 6. Train the Model

In [ ]:
history = model.fit(
    X_train,
    Y_train,
    validation_split=0.1,
    epochs=5,
    batch_size=128
)

## 7. Evaluate the Model

In [ ]:
test_loss, test_accuracy = model.evaluate(X_test, Y_test, verbose=0)
print("Test Accuracy:", round(test_accuracy, 4))
print("Test Loss:", round(test_loss, 4))

In [ ]:
plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

## 8. Confusion Matrix

In [ ]:
Y_pred = model.predict(X_test)
Y_pred_labels = np.argmax(Y_pred, axis=1)
cm = confusion_matrix(Y_test, Y_pred_labels)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()

## 9. Save the Trained Model

In [ ]:
from pathlib import Path

Path("models").mkdir(exist_ok=True)
model.save("models/mnist_model.keras")
print("Model saved successfully.")

## 10. Predict a Custom Digit Image

In [ ]:
def prepare_custom_image(image_path):
    image = Image.open(image_path).convert("L")
    image = ImageOps.autocontrast(image)
    if np.asarray(image).mean() > 127:
        image = ImageOps.invert(image)
    image = image.resize((28, 28), Image.Resampling.LANCZOS)
    image_array = np.asarray(image).astype("float32") / 255.0
    return image_array.reshape(1, 28, 28)

custom_image = prepare_custom_image("3-digit.PNG")
prediction = model.predict(custom_image)
predicted_label = np.argmax(prediction)
print("The handwritten digit is recognized as:", predicted_label)

## Conclusion

The neural network successfully learns patterns from handwritten digit images and can classify new digit images. This project demonstrates dataset loading, preprocessing, model training, evaluation, and prediction.